# 02 — Gold audit (oracle-gold vs oracle-exec)

Run `musicinstruct audit-gold` on the test split. **CPU OK.**

Results: `MyDrive/midi-agent/runs/<RUN_ID>/audit_gold/`.

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1" if DATASET_MODE == "real_v0.2" else "pilot_v1"  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "validation", "train"]
MAX_ITEMS = None  # @param {type:"raw"}

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_MI = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_MI}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_MI}/data/v0.2"
DRIVE_AGENT = "/content/drive/MyDrive/midi-agent"
DRIVE_AUDIT = f"{DRIVE_AGENT}/runs/{RUN_ID}/audit_gold"
LOCAL_AUDIT = f"{WORK}/audit_gold"

In [ ]:
# @title Bootstrap
import shutil
import subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
subprocess.run(f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}", shell=True, check=True)
subprocess.run(f"pip install -q -e {MUSICINSTRUCT}", shell=True, check=True)

Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
if Path(LOCAL_AUDIT).exists():
    shutil.rmtree(LOCAL_AUDIT)

In [ ]:
# @title Run audit-gold
import subprocess

cmd = ["musicinstruct", "audit-gold", MANIFEST, "--output-dir", LOCAL_AUDIT, "--split", SPLIT]
if MAX_ITEMS:
    cmd.extend(["--max-items", str(int(MAX_ITEMS))])
subprocess.run(cmd, check=True)

In [ ]:
# @title Summarize & sync
import json
import shutil
from pathlib import Path

report = json.loads(Path(LOCAL_AUDIT).joinpath("audit_report.json").read_text())
print("oracle-gold:", report["overall_gold"])
print("oracle-exec:", report["overall_exec"])
print("bucket_counts:", report["bucket_counts"])

if USE_DRIVE:
    Path(DRIVE_AUDIT).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_AUDIT, DRIVE_AUDIT, dirs_exist_ok=True)
    print("Saved to", DRIVE_AUDIT)